In [1]:
import requests
from bs4 import BeautifulSoup, NavigableString
import pandas as pd
import time
import random
import re

In [2]:
BASE_URL = "https://www.novelupdates.com/series-ranking/?rank=month&ge=168,324,18,157,851,12,1692,281,560,922&rl=0&pg={}"

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)",
    "Mozilla/5.0 (X11; Linux x86_64)"
]

session = requests.Session()


In [3]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = text.replace("<<less", "").replace("more>>", "")
    return text.strip()


In [4]:
def safe_request(url, max_try=10):
    for i in range(max_try):
        try:
            headers = {
                "User-Agent": random.choice(USER_AGENTS),
                "Accept-Language": "en-US,en;q=0.9",
                "Referer": "https://www.novelupdates.com/"
            }

            res = session.get(url, headers=headers, timeout=10)

            if res.status_code == 200:
                return res

        except Exception as e:
            print(f"[Retry {i+1}] {e}")

        # exponential backoff
        sleep_time = random.uniform(5, 10) + i * 2
        print(f"Sleep {sleep_time:.1f}s before retry...")
        time.sleep(sleep_time)

    return None

In [5]:
def extract_summary(item):
    summary_parts = []

    for content in item.contents:
        if isinstance(content, NavigableString):
            text = content.strip()
            if text:
                summary_parts.append(text)

        elif getattr(content, "attrs", None):
            if "class" in content.attrs and "testhide" in content.attrs["class"]:
                text = content.get_text(" ", strip=True)
                summary_parts.append(text)

    return clean_text(" ".join(summary_parts))


In [6]:
def scrape_clean(max_page=100, save_every=5):
    data = []
    fail_count = 0

    for page in range(1, max_page + 1):
        url = BASE_URL.format(page)
        print(f"\nScraping page {page}...")

        res = safe_request(url)

        if res is None:
            print(f"Page {page} gagal total")
            fail_count += 1

            # stop kalau terlalu banyak gagal
            if fail_count >= 5:
                print("Terlalu banyak gagal, stop scraping dulu.")
                break

            continue
        else:
            fail_count = 0

        soup = BeautifulSoup(res.text, "html.parser")
        items = soup.select(".search_body_nu")

        if len(items) == 0:
            print("No more data, stop.")
            break

        for item in items:
            title_tag = item.select_one(".search_title a")
            title = title_tag.text.strip() if title_tag else ""
            link = title_tag.get("href") if title_tag else ""

            genres = [g.text.strip() for g in item.select(".search_genre a")]
            summary = extract_summary(item)

            data.append({
                "title": title,
                "summary": summary,
                "genres": ", ".join(genres),
                "url": link
            })

        # autosave
        if page % save_every == 0:
            pd.DataFrame(data).to_csv("./dataset/novel_checkpoint.csv", index=False)
            print(f"Checkpoint saved at page {page}")

        # delay normal
        time.sleep(random.uniform(4, 8))

        # cooldown tiap 10 page
        if page % 10 == 0:
            print("Cooling down...")
            time.sleep(random.uniform(20, 40))

    return pd.DataFrame(data)

In [7]:
df = scrape_clean(max_page=200)
df.to_csv("./dataset/novel_dataset_9.csv", index=False)

print("Done! Dataset siap 🚀")


Scraping page 1...

Scraping page 2...

Scraping page 3...

Scraping page 4...

Scraping page 5...
Checkpoint saved at page 5

Scraping page 6...

Scraping page 7...

Scraping page 8...

Scraping page 9...

Scraping page 10...
Checkpoint saved at page 10
Cooling down...

Scraping page 11...

Scraping page 12...

Scraping page 13...

Scraping page 14...

Scraping page 15...
Checkpoint saved at page 15

Scraping page 16...

Scraping page 17...

Scraping page 18...

Scraping page 19...

Scraping page 20...
Checkpoint saved at page 20
Cooling down...

Scraping page 21...

Scraping page 22...

Scraping page 23...

Scraping page 24...

Scraping page 25...
Checkpoint saved at page 25

Scraping page 26...

Scraping page 27...

Scraping page 28...

Scraping page 29...

Scraping page 30...
Checkpoint saved at page 30
Cooling down...

Scraping page 31...

Scraping page 32...

Scraping page 33...

Scraping page 34...

Scraping page 35...
Checkpoint saved at page 35

Scraping page 36...

Scraping 

In [8]:
import pandas as pd
df = pd.read_csv("./dataset/novel_dataset_9.csv")

In [9]:
df

,title,summary,genres,url
0,Semi-Coercive Imperialist,I became a Semi-Coercive Imperialist... To pre...,"Action, Drama, Fantasy, Psychological, Tragedy",https://www.novelupdates.com/series/semi-coerc...
1,"Transmigrated as the Villain, I Let the Dragon...","At the start, I became a rich second-generatio...","Comedy, Harem, Romance, Xuanhuan",https://www.novelupdates.com/series/transmigra...
2,The Villain Who Invests in a Witch to Survive,After transmigrating into a cannon-fodder vill...,"Adventure, Comedy, Fantasy, Harem",https://www.novelupdates.com/series/the-villai...
3,I Was Mistaken for a Saint in a Dark Fantasy,I opened my eyes and it was a dark fantasy wor...,"Comedy, Drama, Fantasy, Harem, Mystery, Psycho...",https://www.novelupdates.com/series/i-was-mist...
4,The Galgame Martial Saint,"“Senior Brother, why has your cultivation impr...","Action, Adventure, Comedy, Fantasy, Harem, Mar...",https://www.novelupdates.com/series/the-galgam...
...,...,...,...,...
4995,Transcendence Through Training,"25-year-old Kim Hyunseok, an orphan abandoned ...","Action, Adventure, Fantasy, Martial Arts, Supe...",https://www.novelupdates.com/series/transcende...
4996,"After Traveling Through the Book, I Had a Happ...","Male supporting role/ Marriage first, love lat...","Adult, Drama, Mature, Psychological, Romance",https://www.novelupdates.com/series/after-trav...
4997,Our Quest ~Climbing to The Top in Another Worl...,"Masamune that died from accident, get his life...","Adventure, Comedy, Fantasy, Harem",https://www.novelupdates.com/series/our-quest-...
4998,"Don’t Be Scared, I’m Really Not a Yandere!","“Yandere… Am I a yandere? Eww, no way! That’s ...","Fantasy, Harem, Romance",https://www.novelupdates.com/series/dont-be-sc...
